<a href="https://colab.research.google.com/github/Srishhtee/PCS221/blob/main/cc_assignment_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Q1. Word Count - Count the frequency of each word
Input:
• hadoop is fast
• hadoop is scalable

In [1]:
data = ["hadoop is fast", "hadoop is scalable"]

Manual

In [2]:
def mapper(data):
    mapped = []
    for line in data:
        for word in line.split():
            mapped.append((word, 1))
    return mapped

In [3]:
from collections import defaultdict

def shuffle(mapped):
    grouped = defaultdict(list)
    for key, value in mapped:
        grouped[key].append(value)
    return grouped

In [4]:
def reducer(grouped):
    return {k: sum(v) for k, v in grouped.items()}

mapped = mapper(data)
grouped = shuffle(mapped)
result = reducer(grouped)

print(result)

{'hadoop': 2, 'is': 2, 'fast': 1, 'scalable': 1}


mrjob

In [5]:
!pip install mrjob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.6/439.6 kB 12.8 MB/s eta 0:00:00


In [6]:
from mrjob.job import MRJob

In [7]:
%%writefile wordcount.py

from mrjob.job import MRJob

class WordCount(MRJob):
    def mapper(self, _, line):
        for word in line.split():
            yield word, 1

    def reducer(self, key, values):
        yield key, sum(values)

if __name__ == "__main__":
    WordCount.run()

Writing wordcount.py


In [8]:
%%writefile input.txt
hadoop is fast
hadoop is scalable

Writing input.txt


In [9]:
!python wordcount.py input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/wordcount.root.20260502.152304.986354
Running step 1 of 1...
job output is in /tmp/wordcount.root.20260502.152304.986354/output
Streaming final output from /tmp/wordcount.root.20260502.152304.986354/output...
"is"	2
"fast"	1
"hadoop"	2
"scalable"	1
Removing temp directory /tmp/wordcount.root.20260502.152304.986354...


Q2. Character Count - Count the frequency of each character (ignore spaces).
Input: big data

manual

In [10]:
data = ["big data"]

def mapper(data):
    mapped = []
    for line in data:
        for ch in line.replace(" ", ""):
            mapped.append((ch, 1))
    return mapped

mapped = mapper(data)
grouped = shuffle(mapped)
result = reducer(grouped)

print(result)

{'b': 1, 'i': 1, 'g': 1, 'd': 1, 'a': 2, 't': 1}


mrjob

In [11]:
%%writefile q2_charcount.py
from mrjob.job import MRJob

class CharCount(MRJob):
    def mapper(self, _, line):
        # Clean line: remove spaces & newline, make lowercase
        line = line.strip().replace(" ", "").lower()

        for ch in line:
            yield ch, 1

    def reducer(self, key, values):
        yield key, sum(values)

if __name__ == "__main__":
    CharCount.run()

Writing q2_charcount.py


In [12]:
%%writefile input2.txt
big data

Writing input2.txt


In [13]:
!python q2_charcount.py input2.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q2_charcount.root.20260502.152305.497307
Running step 1 of 1...
job output is in /tmp/q2_charcount.root.20260502.152305.497307/output
Streaming final output from /tmp/q2_charcount.root.20260502.152305.497307/output...
"t"	1
"b"	1
"d"	1
"a"	2
"g"	1
"i"	1
Removing temp directory /tmp/q2_charcount.root.20260502.152305.497307...


Q3.Average Word Length (Per Word) - Compute the average length of each word.
Input: data science data big data

manual

In [14]:
data = ["data science data big data"]

def mapper(data):
    mapped = []
    for line in data:
        for word in line.split():
            mapped.append((word, len(word)))
    return mapped

def reducer(grouped):
    return {k: sum(v)/len(v) for k, v in grouped.items()}

mapped = mapper(data)
grouped = shuffle(mapped)
result = reducer(grouped)

print(result)

{'data': 4.0, 'science': 7.0, 'big': 3.0}


mrjob

In [15]:
%%writefile q3_avg_word_len.py
from mrjob.job import MRJob

class AvgWordLength(MRJob):

    def mapper(self, _, line):
        for word in line.strip().split():
            yield word.lower(), (len(word), 1)

    def reducer(self, key, values):
        total_len = 0
        total_count = 0

        for length, count in values:
            total_len += length
            total_count += count

        yield key, total_len / total_count

if __name__ == "__main__":
    AvgWordLength.run()

Writing q3_avg_word_len.py


In [16]:
%%writefile input3.txt
data science data big data

Writing input3.txt


In [17]:
!python q3_avg_word_len.py input3.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q3_avg_word_len.root.20260502.152305.997024
Running step 1 of 1...
job output is in /tmp/q3_avg_word_len.root.20260502.152305.997024/output
Streaming final output from /tmp/q3_avg_word_len.root.20260502.152305.997024/output...
"science"	7.0
"big"	3.0
"data"	4.0
Removing temp directory /tmp/q3_avg_word_len.root.20260502.152305.997024...


Q4.Global Average Word Length - Compute the average length of all words.
Input: hadoop mapreduce spark

manual

In [18]:
data = ["hadoop mapreduce spark"]

def mapper(data):
    mapped = []
    for line in data:
        for word in line.split():
            mapped.append(("all", len(word)))
    return mapped

def reducer(grouped):
    total = sum(grouped["all"])
    count = len(grouped["all"])
    return total / count

mapped = mapper(data)
grouped = shuffle(mapped)
result = reducer(grouped)

print(result)

6.666666666666667


mrjob

In [19]:
%%writefile q4_global_avg.py
from mrjob.job import MRJob

class GlobalAvgWordLength(MRJob):

    def mapper(self, _, line):
        for word in line.strip().split():
            # emit (key, (length, count))
            yield "global", (len(word), 1)

    def reducer(self, key, values):
        total_length = 0
        total_count = 0

        for length, count in values:
            total_length += length
            total_count += count

        yield key, total_length / total_count

if __name__ == "__main__":
    GlobalAvgWordLength.run()

Writing q4_global_avg.py


In [20]:
%%writefile input4.txt
hadoop mapreduce spark

Writing input4.txt


In [21]:
!python q4_global_avg.py input4.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q4_global_avg.root.20260502.152306.394904
Running step 1 of 1...
job output is in /tmp/q4_global_avg.root.20260502.152306.394904/output
Streaming final output from /tmp/q4_global_avg.root.20260502.152306.394904/output...
"global"	6.666666666666667
Removing temp directory /tmp/q4_global_avg.root.20260502.152306.394904...


Q5. Perform Q1-Q4 on the file
Also find Top 5 most frequent words

In [23]:
from collections import Counter

with open("/content/shakespeare.txt") as f:
    data = f.readlines()

mapped = []
for line in data:
    for word in line.split():
        mapped.append((word.lower(), 1))

def reducer(grouped):
    result = {}
    for key, values in grouped.items():
        result[key] = sum(values)
    return result


grouped = shuffle(mapped)
word_count = reducer(grouped)

top = Counter(word_count).most_common(5)

print(top)



[('the', 27729), ('and', 26099), ('i', 19540), ('to', 18762), ('of', 18126)]


Q6. Compute average marks for each student.
Input:
A 80
B 70
A 90
B 60
A 100

manual

In [24]:
data = ["A 80", "B 70", "A 90", "B 60", "A 100"]

def mapper(data):
    mapped = []
    for line in data:
        name, marks = line.split()
        mapped.append((name, int(marks)))
    return mapped

def reducer(grouped):
    return {k: sum(v)/len(v) for k, v in grouped.items()}

mapped = mapper(data)
grouped = shuffle(mapped)
result = reducer(grouped)

print(result)

{'A': 90.0, 'B': 65.0}


mrjob

In [25]:
%%writefile q6_avg_marks.py
from mrjob.job import MRJob

class MRAverageMarks(MRJob):

    def mapper(self, _, line):
        student, marks = line.split()
        yield student, int(marks)

    def reducer(self, student, marks):
        marks = list(marks)
        avg = sum(marks) / len(marks)
        yield student, avg

if __name__ == '__main__':
    MRAverageMarks.run()

Writing q6_avg_marks.py


In [26]:
%%writefile input6.txt
A 80
B 70
A 90
B 60
A 100

Writing input6.txt


In [27]:
!python q6_avg_marks.py input6.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q6_avg_marks.root.20260502.152634.937668
Running step 1 of 1...
job output is in /tmp/q6_avg_marks.root.20260502.152634.937668/output
Streaming final output from /tmp/q6_avg_marks.root.20260502.152634.937668/output...
"B"	65.0
"A"	90.0
Removing temp directory /tmp/q6_avg_marks.root.20260502.152634.937668...


Q7.Compute average salary per department and Highest Paid Department (Based on Average
Salary)
Input:
HR 50000
IT 70000
HR 60000
IT 80000

manual

In [28]:
data = ["HR 50000", "IT 70000", "HR 60000", "IT 80000"]

mapped = [(line.split()[0], int(line.split()[1])) for line in data]

grouped = shuffle(mapped)

avg_salary = {k: sum(v)/len(v) for k, v in grouped.items()}

highest = max(avg_salary, key=avg_salary.get)

print("Average:", avg_salary)
print("Highest Paid Dept:", highest)

Average: {'HR': 55000.0, 'IT': 75000.0}
Highest Paid Dept: IT


mrjob

In [29]:
%%writefile q7_salary.py
from mrjob.job import MRJob
from mrjob.step import MRStep

class MRSalaryAnalysis(MRJob):

    def steps(self):
        return [
            MRStep(mapper=self.mapper,
                   reducer=self.reducer_avg),
            MRStep(reducer=self.reducer_max)
        ]

    # Mapper
    def mapper(self, _, line):
        dept, salary = line.split()
        yield dept, int(salary)

    # Step 1: Compute average salary per department
    def reducer_avg(self, dept, salaries):
        salaries = list(salaries)
        avg = sum(salaries) / len(salaries)
        yield None, (dept, avg)

    # Step 2: Find highest paid department
    def reducer_max(self, _, dept_avgs):
        max_dept = None
        max_avg = 0

        for dept, avg in dept_avgs:
            yield dept, avg   # output each dept avg
            if avg > max_avg:
                max_avg = avg
                max_dept = dept

        yield "Highest Paid Department", (max_dept, max_avg)

if __name__ == '__main__':
    MRSalaryAnalysis.run()

Writing q7_salary.py


In [30]:
%%writefile input7.txt
HR 50000
IT 70000
HR 60000
IT 80000

Writing input7.txt


In [31]:
!python q7_salary.py input7.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q7_salary.root.20260502.152645.929988
Running step 1 of 2...
Running step 2 of 2...
job output is in /tmp/q7_salary.root.20260502.152645.929988/output
Streaming final output from /tmp/q7_salary.root.20260502.152645.929988/output...
"IT"	75000.0
"HR"	55000.0
"Highest Paid Department"	["IT", 75000.0]
Removing temp directory /tmp/q7_salary.root.20260502.152645.929988...


Q8.Computer average temperature per country
New York,38
London,29
Tokyo,35
New York,32
Delhi,45
Ambala,35

manual

In [32]:
data = [
    "New York,38", "London,29", "Tokyo,35",
    "New York,32", "Delhi,45", "Ambala,35"
]

mapped = []
for line in data:
    city, temp = line.split(",")
    mapped.append((city, int(temp)))

grouped = shuffle(mapped)

result = {k: sum(v)/len(v) for k, v in grouped.items()}

print(result)

{'New York': 35.0, 'London': 29.0, 'Tokyo': 35.0, 'Delhi': 45.0, 'Ambala': 35.0}


mrjob

In [33]:
%%writefile q8_temp.py
from mrjob.job import MRJob

class MRAvgTemp(MRJob):

    # Mapper
    def mapper(self, _, line):
        city, temp = line.split(',')
        yield city, int(temp)

    # Reducer
    def reducer(self, city, temps):
        temps = list(temps)
        avg = sum(temps) / len(temps)
        yield city, avg

if __name__ == '__main__':
    MRAvgTemp.run()

Writing q8_temp.py


In [34]:
%%writefile input8.txt
New York,38
London,29
Tokyo,35
New York,32
Delhi,45
Ambala,35

Writing input8.txt


In [35]:
!python q8_temp.py input8.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q8_temp.root.20260502.152656.890509
Running step 1 of 1...
job output is in /tmp/q8_temp.root.20260502.152656.890509/output
Streaming final output from /tmp/q8_temp.root.20260502.152656.890509/output...
"London"	29.0
"New York"	35.0
"Ambala"	35.0
"Delhi"	45.0
"Tokyo"	35.0
Removing temp directory /tmp/q8_temp.root.20260502.152656.890509...
